# v6 — Diagnóstico Grad-CAM: ¿Por qué el modelo no detecta melanoma?

**Objetivo:** Usar Grad-CAM para visualizar qué zonas de la imagen activa el modelo al clasificar.
Antes de aplicar mejoras (oversample, class weights), queremos entender *por qué* mel (melanoma)
tiene F1 = 0% en v5.

**Preguntas a responder:**
1. ¿El modelo mira la lesión o mira artefactos (fondo, pelo, regla)?
2. ¿Con qué clase confunde más el melanoma?
3. ¿Hay diferencia visual entre los mel que se predicen correctamente vs los que no?


## 1. Verificar entorno

In [1]:
import os, sys
import numpy as np
import tensorflow as tf
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

print('TensorFlow:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))
print('NumPy:', np.__version__)


TensorFlow: 2.21.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
NumPy: 2.1.3


## 2. Configuración de rutas

In [16]:
# ── Rutas Kaggle ──
NUMPY_DIR  = '/kaggle/input/datasets/michellercarranza/ham10000-numpy'
MODELS_DIR = '/kaggle/input/datasets/michellercarranza/modelos-v5'
OUTPUT_DIR = '/kaggle/working/gradcam_v6'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Rutas de datos
TEST_X_PATH  = os.path.join(NUMPY_DIR, 'test_X.npy')
TEST_Y_PATH  = os.path.join(NUMPY_DIR, 'test_y.npy')

# Nombre del modelo v5
MODEL_PATH = os.path.join(MODELS_DIR, 'densenet201_v5_phase2b_best.keras')

CLASS_NAMES = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
MEL_IDX = CLASS_NAMES.index('mel')

print('MODEL_PATH existe:', os.path.exists(MODEL_PATH))
print('TEST_X existe:   ', os.path.exists(TEST_X_PATH))
print('TEST_Y existe:   ', os.path.exists(TEST_Y_PATH))

MODEL_PATH existe: True
TEST_X existe:    True
TEST_Y existe:    True


In [17]:
import os
for root, dirs, files in os.walk('/kaggle/input'):
    for d in dirs:
        print(os.path.join(root, d))
    for f in files:
        print(os.path.join(root, f))

/kaggle/input/datasets
/kaggle/input/notebooks
/kaggle/input/datasets/michellercarranza
/kaggle/input/datasets/michellercarranza/ham10000-numpy
/kaggle/input/datasets/michellercarranza/modelos-v5
/kaggle/input/datasets/michellercarranza/ham10000-numpy/test_X.npy
/kaggle/input/datasets/michellercarranza/ham10000-numpy/test_y.npy
/kaggle/input/datasets/michellercarranza/ham10000-numpy/train_X.npy
/kaggle/input/datasets/michellercarranza/ham10000-numpy/train_y.npy
/kaggle/input/datasets/michellercarranza/ham10000-numpy/validation_X.npy
/kaggle/input/datasets/michellercarranza/ham10000-numpy/validation_y.npy
/kaggle/input/datasets/michellercarranza/modelos-v5/densenet201_v5_phase2b_best.keras
/kaggle/input/notebooks/michellercarranza


In [18]:
# Verificar archivos disponibles en MODELS_DIR
for root, dirs, files in os.walk(MODELS_DIR):
    for f in files:
        fp = os.path.join(root, f)
        print(fp, f'  ({os.path.getsize(fp)/1e6:.1f} MB)')


/kaggle/input/datasets/michellercarranza/modelos-v5/densenet201_v5_phase2b_best.keras   (94.7 MB)


## 3. Cargar datos de test

In [20]:
print('Cargando test set...')
X_test = np.load(TEST_X_PATH)   # (1285, 224, 224, 3)
y_test = np.load(TEST_Y_PATH)   # (1285,) int32 — ya son enteros

print(f'X_test shape: {X_test.shape}  dtype: {X_test.dtype}')
print(f'y_test shape: {y_test.shape}  dtype: {y_test.dtype}')

# y_test ya es int, no necesita argmax
y_test_int = y_test

# Distribución en test
for i, name in enumerate(CLASS_NAMES):
    n = np.sum(y_test_int == i)
    print(f'  {name:6s}: {n:4d} imágenes')

Cargando test set...
X_test shape: (1285, 224, 224, 3)  dtype: float32
y_test shape: (1285,)  dtype: int32
  akiec :   42 imágenes
  bcc   :   67 imágenes
  bkl   :  142 imágenes
  df    :   14 imágenes
  mel   :  144 imágenes
  nv    :  858 imágenes
  vasc  :   18 imágenes


In [21]:
# Índices de melanoma en test
mel_indices = np.where(y_test_int == MEL_IDX)[0]
print(f'Muestras de melanoma en test: {len(mel_indices)}')

X_mel = X_test[mel_indices]
y_mel = y_test[mel_indices]
print(f'X_mel shape: {X_mel.shape}')


Muestras de melanoma en test: 144
X_mel shape: (144, 224, 224, 3)


## 4. Cargar modelo v5

In [22]:
print('Cargando modelo v5...')

# Registrar Focal Loss para carga correcta
class FocalLoss(tf.keras.losses.Loss):
    def __init__(self, gamma=2.0, alpha=None, **kwargs):
        super().__init__(**kwargs)
        self.gamma = gamma
        self.alpha = alpha

    def call(self, y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        ce = -y_true * tf.math.log(y_pred)
        pt = tf.reduce_sum(y_true * y_pred, axis=-1, keepdims=True)
        focal_weight = tf.pow(1.0 - pt, self.gamma)
        loss = focal_weight * ce
        return tf.reduce_mean(tf.reduce_sum(loss, axis=-1))

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'gamma': self.gamma, 'alpha': self.alpha})
        return cfg

model = tf.keras.models.load_model(
    MODEL_PATH,
    custom_objects={'FocalLoss': FocalLoss}
)
print('Modelo cargado.')
print(f'Input shape: {model.input_shape}')
print(f'Output shape: {model.output_shape}')


Cargando modelo v5...
Modelo cargado.
Input shape: (None, 224, 224, 3)
Output shape: (None, 7)


In [27]:
densenet_layer = model.get_layer('densenet201')

conv_layers = [l for l in densenet_layer.layers 
               if 'conv' in l.name.lower()]

print(f'Capas conv dentro de DenseNet201: {len(conv_layers)}')
print(f'Última capa conv: {conv_layers[-1].name}')

GRADCAM_LAYER = conv_layers[-1].name
print(f'Capa usada para Grad-CAM: {GRADCAM_LAYER}')

Capas conv dentro de DenseNet201: 692
Última capa conv: conv5_block32_concat
Capa usada para Grad-CAM: conv5_block32_concat


## 5. Evaluación del modelo en test completo

In [28]:
print('Evaluando en test set completo...')
y_pred_probs = model.predict(X_test, batch_size=64, verbose=1)
y_pred_int = np.argmax(y_pred_probs, axis=1)

print('\nReporte de clasificación:')
print(classification_report(y_test_int, y_pred_int, target_names=CLASS_NAMES))


Evaluando en test set completo...
21/21 ━━━━━━━━━━━━━━━━━━━━ 64s 2s/step 

Reporte de clasificación:
              precision    recall  f1-score   support

       akiec       1.00      0.02      0.05        42
         bcc       0.41      0.69      0.51        67
         bkl       0.33      0.65      0.44       142
          df       1.00      0.21      0.35        14
         mel       0.00      0.00      0.00       144
          nv       0.85      0.85      0.85       858
        vasc       0.55      0.94      0.69        18

    accuracy                           0.69      1285
   macro avg       0.59      0.48      0.41      1285
weighted avg       0.68      0.69      0.66      1285



/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [29]:
# Matriz de confusión
cm = confusion_matrix(y_test_int, y_pred_int)

plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='OrRd',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            linewidths=0.5)
plt.title('Matriz de confusión — v5 Test Set', fontsize=14, pad=12)
plt.ylabel('Clase real', fontsize=11)
plt.xlabel('Clase predicha', fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrix_v5.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Guardada: confusion_matrix_v5.png')


Guardada: confusion_matrix_v5.png


In [30]:
# Zoom en la fila de mel: ¿con qué clase se confunde?
mel_row = cm[MEL_IDX]
print('\nPredicciones para las muestras de MEL (melanoma):')
for i, (name, count) in enumerate(zip(CLASS_NAMES, mel_row)):
    bar = '█' * count
    marker = ' ← CORRECTO' if i == MEL_IDX else ''
    print(f'  Predicho como {name:6s}: {count:3d}  {bar}{marker}')



Predicciones para las muestras de MEL (melanoma):
  Predicho como akiec :   0  
  Predicho como bcc   :  10  ██████████
  Predicho como bkl   :  46  ██████████████████████████████████████████████
  Predicho como df    :   0  
  Predicho como mel   :   0   ← CORRECTO
  Predicho como nv    :  86  ██████████████████████████████████████████████████████████████████████████████████████
  Predicho como vasc  :   2  ██


## 6. Implementación de Grad-CAM

Grad-CAM calcula los gradientes de la clase objetivo respecto a los mapas de activación
de la última capa convolucional. Las zonas con gradientes altos son las que más influyeron
en la predicción.


In [32]:
import numpy as np
import tensorflow as tf

# Colormap jet en numpy (sin OpenCV)
def apply_jet_colormap(gray):
    """Aplica colormap jet a imagen grayscale [0,1] -> RGB [0,255]"""
    gray = np.clip(gray, 0, 1)
    r = np.clip(1.5 - np.abs(4*gray - 3), 0, 1)
    g = np.clip(1.5 - np.abs(4*gray - 2), 0, 1)
    b = np.clip(1.5 - np.abs(4*gray - 1), 0, 1)
    return np.stack([r, g, b], axis=-1)

def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    densenet = model.get_layer('densenet201')
    grad_model = tf.keras.models.Model(
        inputs=densenet.inputs,
        outputs=[densenet.get_layer(last_conv_layer_name).output, densenet.output]
    )
    with tf.GradientTape() as tape:
        inputs = tf.cast(img_array, tf.float32)
        conv_outputs, predictions = grad_model(inputs)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]
    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

def overlay_gradcam(img, heatmap, alpha=0.4):
    """Superpone el heatmap sobre la imagen original."""
    import cv2
    heatmap_resized = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
    heatmap_rgb = apply_jet_colormap(heatmap_resized)
    heatmap_uint8 = (heatmap_rgb * 255).astype(np.uint8)
    img_uint8 = (img * 255).astype(np.uint8) if img.max() <= 1.0 else img.astype(np.uint8)
    superimposed = (alpha * heatmap_uint8 + (1 - alpha) * img_uint8).astype(np.uint8)
    return superimposed, heatmap_uint8

print("Funciones Grad-CAM definidas correctamente.")

Funciones Grad-CAM definidas correctamente.


In [33]:
# Probar Grad-CAM en una imagen de mel
sample_img = X_mel[0:1]  # (1, 128, 128, 3)
heatmap = make_gradcam_heatmap(sample_img, model, GRADCAM_LAYER, pred_index=MEL_IDX)
print(f'Heatmap shape: {heatmap.shape}  min: {heatmap.min():.3f}  max: {heatmap.max():.3f}')
print('Grad-CAM funcionando correctamente.')


Heatmap shape: (7, 7)  min: 0.104  max: 1.000
Grad-CAM funcionando correctamente.


## 7. Galería Grad-CAM — Muestras de Melanoma

Para cada muestra de mel en test mostramos:
- **Imagen original**
- **Grad-CAM activando para MEL** (qué mira el modelo buscando melanoma)
- **Grad-CAM activando para la clase PREDICHA** (qué mira cuando se equivoca)
- **Probabilidades** de las 7 clases


In [35]:
import cv2

def plot_mel_sample(img, model, gradcam_layer, true_idx, mel_idx, class_names, ax_row):
    """Dibuja una fila: original | cam_mel | cam_predicha | barras proba"""
    # Predicción
    probs = model.predict(img[np.newaxis], verbose=0)[0]
    pred_idx = np.argmax(probs)
    pred_name = class_names[pred_idx]
    correct = pred_idx == true_idx

    # Heatmap para mel (lo que busca)
    hm_mel = make_gradcam_heatmap(img[np.newaxis], model, gradcam_layer, pred_index=mel_idx)
    hm_mel_r = cv2.resize(hm_mel, (img.shape[1], img.shape[0]))
    hm_mel_col = cv2.applyColorMap(np.uint8(255 * hm_mel_r), cv2.COLORMAP_JET)
    hm_mel_col = cv2.cvtColor(hm_mel_col, cv2.COLOR_BGR2RGB) / 255.0
    overlay_mel = np.clip(hm_mel_col * 0.45 + img * 0.55, 0, 1)

    # Heatmap para clase predicha
    hm_pred = make_gradcam_heatmap(img[np.newaxis], model, gradcam_layer, pred_index=pred_idx)
    hm_pred_r = cv2.resize(hm_pred, (img.shape[1], img.shape[0]))
    hm_pred_col = cv2.applyColorMap(np.uint8(255 * hm_pred_r), cv2.COLORMAP_JET)
    hm_pred_col = cv2.cvtColor(hm_pred_col, cv2.COLOR_BGR2RGB) / 255.0
    overlay_pred = np.clip(hm_pred_col * 0.45 + img * 0.55, 0, 1)

    # Columna 1: original
    ax_row[0].imshow(img)
    ax_row[0].set_title(f'Real: mel\nPredicho: {pred_name}',
                        color='green' if correct else 'red', fontsize=8)
    ax_row[0].axis('off')

    # Columna 2: cam para mel
    ax_row[1].imshow(overlay_mel)
    ax_row[1].set_title('Grad-CAM → mel', fontsize=8, color='#8b2500')
    ax_row[1].axis('off')

    # Columna 3: cam para predicha
    ax_row[2].imshow(overlay_pred)
    ax_row[2].set_title(f'Grad-CAM → {pred_name}', fontsize=8,
                        color='green' if correct else '#555')
    ax_row[2].axis('off')

    # Columna 4: barras de probabilidad
    colors = ['#c85a08' if i == mel_idx else ('#1e6e35' if i == pred_idx else '#ccc')
              for i in range(len(class_names))]
    ax_row[3].barh(class_names, probs, color=colors, height=0.6)
    ax_row[3].set_xlim(0, 1)
    ax_row[3].tick_params(labelsize=7)
    ax_row[3].set_title(f'p(mel)={probs[mel_idx]:.2f}', fontsize=8)

print('Función lista.')


Función lista.


In [36]:
# Galería: primeras 20 muestras de mel
N_SHOW = min(20, len(mel_indices))
fig, axes = plt.subplots(N_SHOW, 4, figsize=(14, N_SHOW * 2.6))
fig.suptitle(f'Grad-CAM — Muestras de Melanoma (mel) en Test Set\n'
             f'Rojo = zona activa para esa clase | Verde = predicción correcta | Rojo = error',
             fontsize=11, y=1.001)

for i in range(N_SHOW):
    idx = mel_indices[i]
    plot_mel_sample(X_test[idx], model, GRADCAM_LAYER,
                   true_idx=MEL_IDX, mel_idx=MEL_IDX,
                   class_names=CLASS_NAMES, ax_row=axes[i])
    if (i+1) % 5 == 0:
        print(f'  {i+1}/{N_SHOW} procesadas...')

plt.tight_layout()
out_path = os.path.join(OUTPUT_DIR, 'gradcam_mel_gallery.png')
plt.savefig(out_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'Guardada: gradcam_mel_gallery.png')


Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-1.5980392..1.8208281].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-1.8956583..2.535425].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-1.7906162..1.5125438].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-1.8956583..2.2317834].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-1.0201306..1.4722441].


  5/20 procesadas...


Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-1.9481792..2.0648367].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-2.0007002..2.3959913].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-2.0182073..2.2042704].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-1.8781512..2.5005665].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-1.7731092..1.5942485].


  10/20 procesadas...


Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-2.0665298..1.4373858].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-1.8781512..2.535425].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-1.8081232..1.907974].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-1.8606442..1.6639653].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-1.6998693..1.4722441].


  15/20 procesadas...


Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-1.5280112..2.0474076].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-1.5430065..2.1345534].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-1.8781512..2.535425].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-1.6855742..2.6051416].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-2.117904..2.64].


  20/20 procesadas...
Guardada: gradcam_mel_gallery.png


## 8. Análisis cuantitativo de activaciones

¿El modelo activa zonas centrales (lesión) o periféricas (artefactos)?
Calculamos el **porcentaje de activación en el centro** vs los bordes de cada imagen.


In [37]:
def activation_center_ratio(heatmap, center_fraction=0.5):
    """
    Qué fracción de la activación total está en el centro de la imagen.
    center_fraction: fracción del ancho/alto considerada 'centro'
    """
    h, w = heatmap.shape
    margin_h = int(h * (1 - center_fraction) / 2)
    margin_w = int(w * (1 - center_fraction) / 2)
    center = heatmap[margin_h:h-margin_h, margin_w:w-margin_w]
    total = heatmap.sum() + 1e-8
    return center.sum() / total


print('Calculando ratio centro/borde para muestras de mel...')
ratios_mel_vs_mel = []
ratios_mel_vs_pred = []
pred_classes_for_mel = []

for i, idx in enumerate(mel_indices):
    img = X_test[idx]
    probs = model.predict(img[np.newaxis], verbose=0)[0]
    pred_idx = np.argmax(probs)
    pred_classes_for_mel.append(pred_idx)

    hm_mel = make_gradcam_heatmap(img[np.newaxis], model, GRADCAM_LAYER, pred_index=MEL_IDX)
    hm_pred = make_gradcam_heatmap(img[np.newaxis], model, GRADCAM_LAYER, pred_index=pred_idx)

    hm_mel_r = cv2.resize(hm_mel, (img.shape[1], img.shape[0]))
    hm_pred_r = cv2.resize(hm_pred, (img.shape[1], img.shape[0]))

    ratios_mel_vs_mel.append(activation_center_ratio(hm_mel_r))
    ratios_mel_vs_pred.append(activation_center_ratio(hm_pred_r))

    if (i+1) % 20 == 0:
        print(f'  {i+1}/{len(mel_indices)} procesadas')

ratios_mel_vs_mel = np.array(ratios_mel_vs_mel)
ratios_mel_vs_pred = np.array(ratios_mel_vs_pred)
pred_classes_for_mel = np.array(pred_classes_for_mel)

print(f'\nActivación central promedio (buscando mel):  {ratios_mel_vs_mel.mean():.3f}')
print(f'Activación central promedio (clase predicha): {ratios_mel_vs_pred.mean():.3f}')
print(f'(>0.5 = el modelo mira el centro, <0.5 = mira los bordes)')


Calculando ratio centro/borde para muestras de mel...
  20/144 procesadas
  40/144 procesadas
  60/144 procesadas
  80/144 procesadas
  100/144 procesadas
  120/144 procesadas
  140/144 procesadas

Activación central promedio (buscando mel):  0.358
Activación central promedio (clase predicha): 0.339
(>0.5 = el modelo mira el centro, <0.5 = mira los bordes)


In [38]:
# ¿Con qué clase se confunde mel?
print('Distribución de predicciones para muestras de mel:')
for i, name in enumerate(CLASS_NAMES):
    count = np.sum(pred_classes_for_mel == i)
    if count > 0:
        pct = count / len(pred_classes_for_mel) * 100
        bar = '█' * int(pct / 2)
        marker = ' ← CORRECTO' if i == MEL_IDX else ''
        print(f'  Predicho como {name:6s}: {count:3d} ({pct:5.1f}%)  {bar}{marker}')


Distribución de predicciones para muestras de mel:
  Predicho como bcc   :  10 (  6.9%)  ███
  Predicho como bkl   :  46 ( 31.9%)  ███████████████
  Predicho como nv    :  86 ( 59.7%)  █████████████████████████████
  Predicho como vasc  :   2 (  1.4%)  


In [39]:
# Visualización: histograma de ratios + barras de confusión
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histograma de activación central
axes[0].hist(ratios_mel_vs_mel, bins=15, alpha=0.7, color='#c85a08', label='Grad-CAM → mel')
axes[0].hist(ratios_mel_vs_pred, bins=15, alpha=0.7, color='#4a90d9', label='Grad-CAM → pred')
axes[0].axvline(0.5, color='black', linestyle='--', linewidth=1, label='50% (centro = borde)')
axes[0].set_xlabel('Fracción de activación en zona central')
axes[0].set_ylabel('Número de imágenes')
axes[0].set_title('¿El modelo mira el centro o los bordes?')
axes[0].legend(fontsize=9)

# Barras de confusión de mel
counts = [np.sum(pred_classes_for_mel == i) for i in range(len(CLASS_NAMES))]
colors = ['#1e6e35' if i == MEL_IDX else '#c85a08' for i in range(len(CLASS_NAMES))]
axes[1].barh(CLASS_NAMES, counts, color=colors, height=0.6)
axes[1].set_xlabel('Número de muestras de mel predichas como...')
axes[1].set_title('¿Con qué clase confunde mel el modelo?')
for j, v in enumerate(counts):
    if v > 0:
        axes[1].text(v + 0.5, j, str(v), va='center', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'mel_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Guardada: mel_analysis.png')


Guardada: mel_analysis.png


## 9. Comparativa visual: mel vs nv con Grad-CAM

¿Qué características visuales separan (o no) a mel de nv según el modelo?


In [40]:
# Muestras de nv (predichas correctamente)
nv_idx_val = CLASS_NAMES.index('nv')
nv_indices = np.where((y_test_int == nv_idx_val) & (np.argmax(model.predict(X_test, verbose=0), axis=1) == nv_idx_val))[0]
print(f'nv correctamente clasificados: {len(nv_indices)}')

N_COMP = 6  # pares mel vs nv
fig, axes = plt.subplots(N_COMP, 6, figsize=(18, N_COMP * 3))
fig.suptitle('Comparativa Grad-CAM: Melanoma (mel) vs Nevo (nv)\n'
             'Columnas: mel original | cam mel→mel | cam mel→pred || nv original | cam nv→nv | cam nv→pred',
             fontsize=10, y=1.001)

for row in range(N_COMP):
    # MEL
    mel_img = X_test[mel_indices[row]]
    mel_probs = model.predict(mel_img[np.newaxis], verbose=0)[0]
    mel_pred = np.argmax(mel_probs)

    hm_mm = cv2.resize(make_gradcam_heatmap(mel_img[np.newaxis], model, GRADCAM_LAYER, MEL_IDX),
                       (mel_img.shape[1], mel_img.shape[0]))
    hm_mp = cv2.resize(make_gradcam_heatmap(mel_img[np.newaxis], model, GRADCAM_LAYER, mel_pred),
                       (mel_img.shape[1], mel_img.shape[0]))

    def to_overlay(img, hm):
        h = cv2.applyColorMap(np.uint8(255 * hm), cv2.COLORMAP_JET)
        h = cv2.cvtColor(h, cv2.COLOR_BGR2RGB) / 255.0
        return np.clip(h * 0.45 + img * 0.55, 0, 1)

    axes[row][0].imshow(mel_img)
    axes[row][0].set_title(f'MEL → pred:{CLASS_NAMES[mel_pred]}', fontsize=7,
                           color='green' if mel_pred==MEL_IDX else 'red')
    axes[row][0].axis('off')
    axes[row][1].imshow(to_overlay(mel_img, hm_mm))
    axes[row][1].set_title('CAM→mel', fontsize=7)
    axes[row][1].axis('off')
    axes[row][2].imshow(to_overlay(mel_img, hm_mp))
    axes[row][2].set_title(f'CAM→{CLASS_NAMES[mel_pred]}', fontsize=7)
    axes[row][2].axis('off')

    # NV
    nv_img = X_test[nv_indices[row]]
    nv_probs = model.predict(nv_img[np.newaxis], verbose=0)[0]
    nv_pred = np.argmax(nv_probs)

    hm_nm = cv2.resize(make_gradcam_heatmap(nv_img[np.newaxis], model, GRADCAM_LAYER, nv_idx_val),
                       (nv_img.shape[1], nv_img.shape[0]))
    hm_np = cv2.resize(make_gradcam_heatmap(nv_img[np.newaxis], model, GRADCAM_LAYER, nv_pred),
                       (nv_img.shape[1], nv_img.shape[0]))

    axes[row][3].imshow(nv_img)
    axes[row][3].set_title(f'NV → pred:{CLASS_NAMES[nv_pred]}', fontsize=7, color='green')
    axes[row][3].axis('off')
    axes[row][4].imshow(to_overlay(nv_img, hm_nm))
    axes[row][4].set_title('CAM→nv', fontsize=7)
    axes[row][4].axis('off')
    axes[row][5].imshow(to_overlay(nv_img, hm_np))
    axes[row][5].set_title(f'CAM→{CLASS_NAMES[nv_pred]}', fontsize=7)
    axes[row][5].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'mel_vs_nv_gradcam.png'), dpi=120, bbox_inches='tight')
plt.show()
print('Guardada: mel_vs_nv_gradcam.png')


nv correctamente clasificados: 728


Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-1.5980392..1.8208281].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-1.7906162..1.8036647].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-1.8956583..2.535425].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-1.230392..1.7351656].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-1.7906162..1.5125438].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-1.6505601..1.9749122].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-1.8956583.

Guardada: mel_vs_nv_gradcam.png


## 10. Diagnóstico final y recomendaciones para v6

Basado en los resultados de Grad-CAM:


In [41]:
print('=' * 60)
print('DIAGNÓSTICO GRAD-CAM — MELANOMA')
print('=' * 60)

mean_center_mel = ratios_mel_vs_mel.mean()
mean_center_pred = ratios_mel_vs_pred.mean()

print(f'\nActivación central (Grad-CAM → mel):  {mean_center_mel:.3f}')
print(f'Activación central (Grad-CAM → pred): {mean_center_pred:.3f}')

if mean_center_mel > 0.55:
    print('\n✅ El modelo SÍ mira el centro de la imagen (la lesión).')
    print('   El problema es de discriminación de características, no de localización.')
    print('   Recomendación: oversample mel + class weight más agresivo (Opciones A+B).')
else:
    print('\n⚠️  El modelo NO mira consistentemente el centro.')
    print('   Puede estar mirando artefactos (pelo, fondo, regla de escala).')
    print('   Recomendación: añadir preprocessing (hair removal) antes de A+B.')

top_confusion = CLASS_NAMES[np.bincount(pred_classes_for_mel[pred_classes_for_mel != MEL_IDX]).argmax()]
print(f'\nMayor fuente de confusión: mel → {top_confusion}')
print(f'Estrategia específica: separar mel de {top_confusion} en espacio de features.')

print('\n' + '=' * 60)
print('Archivos generados en', OUTPUT_DIR)
for f in os.listdir(OUTPUT_DIR):
    fp = os.path.join(OUTPUT_DIR, f)
    print(f'  {f}  ({os.path.getsize(fp)/1e3:.0f} KB)')


DIAGNÓSTICO GRAD-CAM — MELANOMA

Activación central (Grad-CAM → mel):  0.358
Activación central (Grad-CAM → pred): 0.339

⚠️  El modelo NO mira consistentemente el centro.
   Puede estar mirando artefactos (pelo, fondo, regla de escala).
   Recomendación: añadir preprocessing (hair removal) antes de A+B.

Mayor fuente de confusión: mel → nv
Estrategia específica: separar mel de nv en espacio de features.

Archivos generados en /kaggle/working/gradcam_v6
  confusion_matrix_v5.png  (67 KB)
  gradcam_mel_gallery.png  (5805 KB)
  mel_vs_nv_gradcam.png  (4045 KB)
  mel_analysis.png  (68 KB)
